# ema-first-moment — ex1: Adam m-buffer EMA update m = beta1*m + (1-beta1)*g

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `ema-first-moment`. Running the final beacon cell reports progress against the `Optimizer: Adam EMA first moment` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Optimizer: Adam EMA first moment` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`ema-first-moment`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "ema-first-moment"
DD_SUBTOPIC = "Optimizer: Adam EMA first moment"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Adam EMA first moment `m = beta1*m + (1-beta1)*g` — quick refresher

Adam maintains TWO running averages per parameter. The first moment `m` is an EMA of the gradient itself; the second moment `v` (separate drill) is an EMA of the squared gradient. The first-moment recurrence:

```
m_t = beta1 * m_{t-1} + (1 - beta1) * g_t
```

**Why an EMA of `g`, not `g` itself.** The raw gradient is noisy — it swings step-to-step even on a stationary loss surface. The EMA low-pass-filters that noise. With `beta1 = 0.9` (Adam default) the effective averaging window is ~10 recent steps.

**Relation to classical momentum.** Classical momentum is `b_t = mu * b_{t-1} + g_t` (no `(1-mu)` factor). Adam's first moment is the same idea but RESCALED so `m_t` stays on the same magnitude scale as `g_t` — which is what makes the bias-correction divide work cleanly.

**Why we update IN PLACE via `m.copy_(...)`.** The buffer lives in the optimizer's `self.m` list; rebinding `m = beta1*m + (1-beta1)*g` would only update the local for-loop variable, not the list entry. Next step would see the stale zero buffer.

### Exercise 1 — Adam m-buffer EMA update m = beta1*m + (1-beta1)*g

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply the Adam first-moment recurrence `m = beta1*m + (1-beta1)*g` via `buffer.copy_()` so the gradient-EMA buffer state is correctly mutated in place across steps.
> Keywords: adam, first-moment, ema, gradient-ema
> ```

**KCs targeted:** `ema-first-moment-recurrence`, `buffer-copy_-mutates-state-in-place`

Implement `ex1_ema_m_step(m_list, grad_list, beta1)`. The first-moment update from Adam.

For each `(m, g)` pair drawn from `(m_list, grad_list)`:

1. Compute the new value: `beta1 * m + (1 - beta1) * g`.
2. Mutate the buffer IN PLACE: `m.copy_(...)`. Don't rebind.
3. Append the new buffer value (by reference) to the return list.

Inputs:
- `m_list`: list of per-param first-moment buffers (mutated).
- `grad_list`: list of per-param gradients (NOT mutated).
- `beta1`: float in `(0, 1)` — Adam default is `0.9`.

Output: list of updated `m` tensors.

The test runs three steps with KNOWN gradients (including negative values — unlike the second-moment EMA, the first-moment EMA preserves sign) and verifies the buffer is in-place mutated (id and data_ptr preserved across steps).

In [ ]:
def ex1_ema_m_step(m_list: list, grad_list: list, beta1: float) -> list:
    """In-place update: m.copy_(beta1*m + (1-beta1)*g)."""
    raise NotImplementedError()


def _test_ex1():
    # One param, zero-init buffer.
    m = t.zeros(4)
    orig_id = id(m)
    orig_ptr = m.data_ptr()

    # === Step 1: zero buffer + g => m_1 = (1 - beta1) * g ===
    g1 = t.tensor([1.0, -2.0, 3.0, -4.0])
    beta1 = 0.9
    out1 = ex1_ema_m_step([m], [g1], beta1=beta1)
    expected1 = (1 - beta1) * g1
    assert t.allclose(out1[0], expected1), (
        f'step 1: expected {expected1}, got {out1[0]}; '
        f'check formula: beta1*m + (1-beta1)*g'
    )
    assert t.allclose(m, expected1), 'step 1: buffer not mutated to new value'
    assert id(m) == orig_id, 'buffer was rebound — use m.copy_(...) not m = ...'
    assert m.data_ptr() == orig_ptr, 'buffer storage reallocated'

    # === Step 2: same g; m approaches g monotonically (preserving sign) ===
    m_before_step2 = m.clone()
    ex1_ema_m_step([m], [g1], beta1=beta1)
    expected2 = beta1 * expected1 + (1 - beta1) * g1
    assert t.allclose(m, expected2), (
        f'step 2: expected {expected2}, got {m}; '
        f'this fails if step 1 did NOT mutate the buffer (rebind bug)'
    )
    # Sign preservation: m[1] and m[3] negative; m[0] and m[2] positive.
    assert m[0] > 0 and m[2] > 0, f'positive-g coords should be positive: {m}'
    assert m[1] < 0 and m[3] < 0, f'negative-g coords should be negative: {m}'
    # Magnitude growing toward |g|.
    assert (m.abs() > m_before_step2.abs()).all(), (
        'EMA magnitude should grow toward |g| with constant g; '
        'if it shrunk, your formula has the wrong sign'
    )

    # === Step 3 — still moving toward g, never overshooting ===
    ex1_ema_m_step([m], [g1], beta1=beta1)
    # m_3 = (1 - beta1**3) * g for constant g.
    expected3 = (1 - beta1 ** 3) * g1
    assert t.allclose(m, expected3, atol=1e-6), (
        f'step 3 closed-form: expected {expected3}, got {m}'
    )
    # Magnitude is below |g|.
    assert (m.abs() < g1.abs()).all(), 'm must not exceed g in magnitude with constant g'

    # === Multi-param batch ===
    m_multi = [t.zeros(2), t.zeros(3, 3)]
    g_multi = [t.tensor([1.0, -1.0]), t.ones(3, 3) * 2.0]
    ex1_ema_m_step(m_multi, g_multi, beta1=0.5)
    # 0.5 * 0 + 0.5 * [1, -1] = [0.5, -0.5]
    assert t.allclose(m_multi[0], t.tensor([0.5, -0.5])), (
        f'multi-param step: m_multi[0]={m_multi[0]}'
    )
    # 0.5 * 0 + 0.5 * 2 = 1.0 everywhere
    assert t.allclose(m_multi[1], t.ones(3, 3)), (
        f'multi-param step: m_multi[1]={m_multi[1]}'
    )

    # === beta1 = 0 collapses to plain g ===
    m_fresh = t.zeros(3)
    g_fresh = t.tensor([7.0, 8.0, 9.0])
    ex1_ema_m_step([m_fresh], [g_fresh], beta1=0.0)
    assert t.allclose(m_fresh, g_fresh), 'beta1=0: m should just equal g'

    # === Input grad must not be mutated ===
    g_in = t.tensor([2.0, 3.0])
    g_snap = g_in.clone()
    ex1_ema_m_step([t.zeros(2)], [g_in], beta1=0.99)
    assert t.equal(g_in, g_snap), 'grad tensors must not be mutated by the EMA update'
    _dd_passed.add('ex1')
    print("ex1 ✓")

_test_ex1()

<details><summary>Solution</summary>

```python
def ex1_ema_m_step(m_list, grad_list, beta1):
    out = []
    for m, g in zip(m_list, grad_list):
        m.copy_(beta1 * m + (1 - beta1) * g)
        out.append(m)
    return out
```

**Why this is its own atom (separate from `ema-second-moment`).** Numerically the recurrences look symmetric: `m = b1*m + (1-b1)*g` vs `v = b2*v + (1-b2)*g.pow(2)`. But they behave differently: `m` preserves the SIGN of `g` (so it captures direction); `v` is always non-negative (so it captures magnitude). Conflating them is the #2 Adam bug after the rebind bug.

**What the EMA converges to.** For constant `g`, `m_t = (1 - beta1^t) * g`. As `t -> inf`, `m_t -> g`. With `beta1 = 0.9` the effective averaging window is ~10 steps — shorter than the second-moment's ~1000-step window. This is intentional: gradient DIRECTION needs to react faster than gradient MAGNITUDE.

**Why we return `m` by reference.** After `m.copy_(...)`, the buffer IS the new value. Returning `m` (rather than the computed expression) means downstream code that does `theta -= lr * m_hat / (v_hat.sqrt() + eps)` reads from the freshly-mutated buffer.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()